In [3]:
import re
from pathlib import Path
import subprocess, sys
from pathlib import Path

In [ ]:
# -*- coding: utf-8 -*-
# =============================================================================
# CONFIGURACION:
# =============================================================================
RAIZ = Path(r'C:\Users\USUARIO\OneDrive - Universidad Nacional de Colombia'
            r'\Tesis\Escritorio\Códigos\Modelos')

DIAG = {
    "M1": RAIZ / r"Modelo 1\Graficas_de_analisis\diag_m1_py",
    "M2": RAIZ / r"Modelo 2\diag_m2_py",
    "M3": RAIZ / r"Modelo 3\2_Resultados_diagnosticos_python",
    "M4": RAIZ / r"Modelo 4\diagnosticos_M4",
}
MCMC = {
    "M1": RAIZ / r"Modelo 1\mcmc_salida_m1",
    "M2": RAIZ / r"Modelo 2\mcmc_salida_m2",
    "M3": RAIZ / r"Modelo 3\mcmc_salida_m3",
    "M4": RAIZ / r"Modelo 4\export_python",
}

REQUERIDOS = {
    "M1": {
        "critico":    ["resumen_convergencia_M1.csv"],
        "importante": ["ajuste_M1.csv", "khat_M1.csv",
                       "ppc_cobertura_M1.csv", "ppc_estadisticos_M1.csv"],
        "opcional":   ["resumen_ejecutivo_M1.csv"],
    },
    "M2": {
        "critico":    ["resumen_convergencia_M2.csv"],
        "importante": ["ajuste_M2.csv", "metricas_M2.json", "khat_M2.csv",
                       "ppc_estadisticos_M2.csv"],
        "opcional":   [],
    },
    "M3": {
        "critico":    ["convergencia_resumen.csv"],
        "importante": ["comparacion_modelo.csv", "reporte_diagnostico_M3.md"],
        "opcional":   ["dic_M3.csv", "ajuste_dic_M3.csv"],
    },
    "M4": {
        "critico":    ["tabla_convergencia_monitoreados.csv",
                       "tabla_convergencia_theta.csv"],
        "importante": ["tabla_ajuste_waic.csv", "tabla_ppc.csv",
                       "tabla_cobertura.csv", "resumen_diagnostico_M4.txt"],
        "opcional":   [],
    },
}


# =============================================================================
def variantes(nombre):
    """Misma logica que _variantes_nombre del comparador: con y sin sufijo."""
    p = Path(nombre)
    tallo, ext = p.stem, p.suffix
    nombres = [p.name]
    m = re.search(r"_(M[1-4])$", tallo, flags=re.IGNORECASE)
    base = tallo[: m.start()] if m else tallo
    if m:
        nombres.append(f"{base}{ext}")
    for suf in ("M1", "M2", "M3", "M4"):
        nombres.append(f"{base}_{suf}{ext}")
    vistos, out = set(), []
    for n in nombres:
        if n.lower() not in vistos:
            vistos.add(n.lower())
            out.append(n)
    return out


def resolver(carpeta, nombre):
    """Primera variante existente dentro de la carpeta, o None."""
    carpeta = Path(carpeta)
    if not carpeta.is_dir():
        return None
    presentes = {f.name.lower(): f for f in carpeta.iterdir() if f.is_file()}
    for v in variantes(nombre):
        if v.lower() in presentes:
            return presentes[v.lower()]
    return None


def buscar_en_arbol(raiz, nombre, modelo, tope=8):
    """Busca el archivo en todo el arbol bajo raiz.

    A diferencia de resolver(), aqui NO se aceptan los sufijos de los demas
    modelos: buscando resumen_convergencia_M1.csv no tiene ningun sentido
    proponer resumen_convergencia_M2.csv, que corresponde a otro modelo y
    llevaria a copiar el archivo equivocado.
    """
    p = Path(nombre)
    tallo, ext = p.stem, p.suffix
    m = re.search(r"_(M[1-4])$", tallo, flags=re.IGNORECASE)
    base = tallo[: m.start()] if m else tallo
    objetivo = {f"{base}{ext}".lower(), f"{base}_{modelo}{ext}".lower(),
                p.name.lower()}
    hallados = []
    try:
        for f in Path(raiz).rglob("*"):
            if f.is_file() and f.name.lower() in objetivo:
                hallados.append(f)
                if len(hallados) >= tope:
                    break
    except (OSError, PermissionError):
        pass
    return hallados


# =============================================================================
def main():
    print("=" * 78)
    print("VERIFICACION PREVIA")
    print("=" * 78)
    problemas_criticos = 0
    faltantes = []

    for M in ("M1", "M2", "M3", "M4"):
        d = DIAG[M]
        print(f"\n[{M}] {d}")
        if not d.is_dir():
            print("   *** LA CARPETA NO EXISTE. El script omitiria este "
                  "modelo en silencio. ***")
            problemas_criticos += 1
            continue
        n_files = sum(1 for f in d.iterdir() if f.is_file())
        print(f"   {n_files} archivos en la carpeta")
        for nivel in ("critico", "importante", "opcional"):
            for nombre in REQUERIDOS[M][nivel]:
                hit = resolver(d, nombre)
                if hit:
                    marca = "ok  "
                    detalle = hit.name
                else:
                    marca = {"critico": "FALTA", "importante": "falta",
                             "opcional": "-   "}[nivel]
                    detalle = ""
                    if nivel != "opcional":
                        faltantes.append((M, nombre))
                    if nivel == "critico":
                        problemas_criticos += 1
                print(f"   [{marca:5}] {nivel:10} {nombre:38} {detalle}")

        mc = MCMC[M]
        if not mc.is_dir():
            print(f"   *** MCMC NO EXISTE: {mc}  -> sin capa uniforme ***")
            problemas_criticos += 1
        else:
            nb = sum(1 for f in mc.rglob("*") if f.is_file())
            print(f"   [ok   ] mcmc       {mc.name:38} {nb} archivos")

    # -------- localizar lo que falta ----------------------------------------
    if faltantes:
        print("\n" + "=" * 78)
        print("BUSQUEDA DE LOS ARCHIVOS FALTANTES BAJO LA RAIZ")
        print("=" * 78)
        for M, nombre in faltantes:
            print(f"\n{M} :: {nombre}")
            hits = buscar_en_arbol(RAIZ, nombre, M)
            if not hits:
                print("   no aparece en ninguna parte del arbol")
            for h in hits:
                try:
                    rel = h.relative_to(RAIZ)
                except ValueError:
                    rel = h
                print(f"   -> {rel}")
                print(f"      carpeta: {h.parent}")

    # -------- veredicto ------------------------------------------------------
    print("\n" + "=" * 78)
    if problemas_criticos:
        print(f"NO LANZAR: {problemas_criticos} problema(s) critico(s).")
        print("Recuerda que --mX admite UNA sola carpeta: si los archivos")
        print("estan repartidos, hay que copiarlos todos a un mismo sitio.")
    elif faltantes:
        print("Se puede lanzar, pero faltan artefactos no criticos.")
        print("Revisa la lista de arriba antes de decidir.")
    else:
        print("TODO EN ORDEN. Se puede lanzar la corrida.")
    print("=" * 78)
    return 1 if problemas_criticos else 0


if __name__ == "__main__":
    raise SystemExit(main())

VERIFICACION PREVIA

[M1] C:\Users\USUARIO\OneDrive - Universidad Nacional de Colombia\Tesis\Escritorio\Códigos\Modelos\Modelo 1\Graficas_de_analisis\diag_m1_py
   31 archivos en la carpeta
   [ok   ] critico    resumen_convergencia_M1.csv            resumen_convergencia_M1.csv
   [ok   ] importante ajuste_M1.csv                          ajuste_M1.csv
   [ok   ] importante khat_M1.csv                            khat_M1.csv
   [ok   ] importante ppc_cobertura_M1.csv                   ppc_cobertura_M1.csv
   [ok   ] importante ppc_estadisticos_M1.csv                ppc_estadisticos_M1.csv
   [-    ] opcional   resumen_ejecutivo_M1.csv               
   [ok   ] mcmc       mcmc_salida_m1                         28 archivos

[M2] C:\Users\USUARIO\OneDrive - Universidad Nacional de Colombia\Tesis\Escritorio\Códigos\Modelos\Modelo 2\diag_m2_py
   36 archivos en la carpeta
   [ok   ] critico    resumen_convergencia_M2.csv            resumen_convergencia_M2.csv
   [ok   ] importante ajuste_M2.c

SystemExit: 0

In [ ]:

pth = Path(r'C:\Users\USUARIO\OneDrive - Universidad Nacional de Colombia'
           r'\Tesis\Escritorio\Códigos\Modelos')

SCRIPT = pth / r"Comparación\comparar_modelos_v18.py"

cmd = [
    sys.executable, SCRIPT,
    "--m1",      pth / r"Modelo 1\Graficas_de_analisis\diag_m1_py",
    "--m1-mcmc", pth / r"Modelo 1\mcmc_salida_m1",
    "--m2",      pth / r"Modelo 2\diag_m2_py",
    "--m2-mcmc", pth / r"Modelo 2\mcmc_salida_m2",
    "--m3",      pth / r"Modelo 3\2_Resultados_diagnosticos_python",
    "--m3-mcmc", pth / r"Modelo 3\mcmc_salida_m3",
    "--m4",      pth / r"Modelo 4\diagnosticos_M4",
    "--m4-mcmc", pth / r"Modelo 4\export_python",
    "--diag",    pth / r"Comparación\salida_6000",   # carpeta NUEVA
    "--draws",     "6000",
    "--bloque-obs", "8000",                          # RAM ~1,5 GB
    "--pdf",                                         # vectorial para LaTeX
]
cmd = [str(x) for x in cmd]

# verificación previa: el script OMITE en silencio cualquier modelo cuya
# carpeta no exista, y terminarías con una comparación de 3 modelos sin notarlo
for i, a in enumerate(cmd):
    if a.startswith("--m") and i + 1 < len(cmd):
        if not Path(cmd[i + 1]).exists():
            raise SystemExit(f"NO EXISTE: {a} -> {cmd[i + 1]}")

with subprocess.Popen(cmd, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True,
                      encoding="utf-8", errors="replace", bufsize=1) as p:
    for linea in p.stdout:
        print(linea, end="", flush=True)
print("código de salida:", p.returncode)